<a href="https://colab.research.google.com/github/NABI-SNU/book/blob/main/tutorials/Session_1_DimensionReduction/Tutorial2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; <a href="https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/NABI-SNU/book/main/tutorials/Session_1_DimensionReduction/Tutorial2.ipynb" target="_parent"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open in Kaggle"/></a>

# Tutorial 2: Visualizing Data with PCA and UMAP

**Session 1: Dimension Reduction**

**Objective:** Compare PCA and UMAP as two-dimensional visualization tools for MNIST.

## Tutorial Objectives

Two-dimensional embeddings are useful for quickly exploring high-dimensional datasets. In this tutorial, we will compare PCA, a linear method, with UMAP, a nonlinear method that often reveals local clusters.

By the end, you will be able to:

- Visualize MNIST in 2D using PCA.
- Visualize MNIST in 2D using UMAP.
- Compare what PCA and UMAP emphasize in the same dataset.
- Explain why UMAP visualizations should be treated as exploratory summaries.

---
# Setup


In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# @title Figure Settings
import logging
logging.getLogger('matplotlib.font_manager').disabled = True

import ipywidgets as widgets  # interactive display
%config InlineBackend.figure_format = 'retina'
plt.style.use("https://raw.githubusercontent.com/NABI-SNU/book/main/nma.mplstyle")

In [ ]:
# @title Plotting Functions

def visualize_components(component1, component2, labels, show=True):
  """
  Plots a 2D representation of the data for visualization with categories
  labelled as different colors.

  Args:
    component1 (numpy array of floats) : Vector of component 1 scores
    component2 (numpy array of floats) : Vector of component 2 scores
    labels (numpy array of floats)     : Vector corresponding to categories of
                                         samples

  Returns:
    Nothing.

  """

  plt.figure()
  plt.scatter(x=component1, y=component2, c=labels, cmap='tab10')
  plt.xlabel('Component 1')
  plt.ylabel('Component 2')
  plt.colorbar(ticks=range(10))
  plt.clim(-0.5, 9.5)
  if show:
    plt.show()

---
# Section 1: Visualize MNIST in 2D using PCA

In this exercise, we'll visualize the first few components of the MNIST dataset to look for evidence of structure in the data. But in this tutorial, we will also be interested in the label of each image (i.e., which numeral it is from 0 to 9). Start by running the following cell to reload the MNIST dataset (this takes a few seconds).

In [ ]:
from sklearn.datasets import fetch_openml

# Get images
mnist = fetch_openml(name='mnist_784', as_frame=False, parser='auto')
X_all = mnist.data

# Get labels
labels_all = np.array([int(k) for k in mnist.target])

**Note:** We saved the complete dataset as `X_all` and the labels as `labels_all`.

To perform PCA, we now will use the method implemented in sklearn. Run the following cell to set the parameters of PCA - we will only look at the top 2 components because we will be visualizing the data in 2D.

In [ ]:
from sklearn.decomposition import PCA

# Initializes PCA
pca_model = PCA(n_components=2)

# Performs PCA
pca_model.fit(X_all)

## Coding Exercise 1: Visualization of MNIST in 2D using PCA

Fill in the code below to perform PCA and visualize the top two components. For faster visualization, take only the first 2,000 samples of the data. We will use the same subset for UMAP in the next section.

**Suggestions:**
- Truncate the data matrix at 2,000 samples. You will also need to truncate the array of labels.
- Perform PCA on the truncated data.
- Use the function `visualize_components` to plot the labeled data.

In [ ]:
help(visualize_components)
help(pca_model.transform)

In [ ]:
#################################################
## TODO for students: take only 2,000 samples and perform PCA
# Comment once you've completed the code
raise NotImplementedError("Student exercise: perform PCA")
#################################################

# Take only the first 2000 samples with the corresponding labels
X, labels = ...

# Perform PCA
scores = pca_model.transform(X)

# Plot the data and reconstruction
visualize_components(...)

---
# Section 2: Visualize MNIST in 2D using UMAP

*Estimated timing to here from start of tutorial: 15 min*


In [ ]:
# @title Video 2: Nonlinear Methods
from ipywidgets import widgets
from IPython.display import YouTubeVideo
from IPython.display import IFrame
from IPython.display import display


class PlayVideo(IFrame):
  def __init__(self, id, source, page=1, width=400, height=300, **kwargs):
    self.id = id
    if source == 'Bilibili':
      src = f'https://player.bilibili.com/player.html?bvid={id}&page={page}'
    elif source == 'Osf':
      src = f'https://mfr.ca-1.osf.io/render?url=https://osf.io/download/{id}/?direct%26mode=render'
    super(PlayVideo, self).__init__(src, width, height, **kwargs)


def display_videos(video_ids, W=400, H=300, fs=1):
  tab_contents = []
  for i, video_id in enumerate(video_ids):
    out = widgets.Output()
    with out:
      if video_ids[i][0] == 'Youtube':
        video = YouTubeVideo(id=video_ids[i][1], width=W,
                             height=H, fs=fs, rel=0)
        print(f'Video available at https://youtube.com/watch?v={video.id}')
      else:
        video = PlayVideo(id=video_ids[i][1], source=video_ids[i][0], width=W,
                          height=H, fs=fs, autoplay=False)
        if video_ids[i][0] == 'Bilibili':
          print(f'Video available at https://www.bilibili.com/video/{video.id}')
        elif video_ids[i][0] == 'Osf':
          print(f'Video available at https://osf.io/{video.id}')
      display(video)
    tab_contents.append(out)
  return tab_contents


video_ids = [('Youtube', '5Xpb0YaN5Ms'), ('Bilibili', 'BV14Z4y1u7HG')]
tab_contents = display_videos(video_ids, W=854, H=480)
tabs = widgets.Tab()
tabs.children = tab_contents
for i in range(len(tab_contents)):
  tabs.set_title(i, video_ids[i][0])
display(tabs)

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Nonlinear_methods_Video")

Next we will analyze the same data using UMAP, a nonlinear dimension reduction method that is useful for visualizing high-dimensional data in 2D or 3D. Run the cell below to get started.

In [ ]:
from umap import UMAP

umap_model = UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=2020)

## Coding Exercise 2.1: Apply UMAP on MNIST
First, we'll run UMAP on the data to explore whether we can see more structure. The cell above defined the parameters that we will use to find our embedding, the low-dimensional representation of the data. To run UMAP on our data, use the function `umap_model.fit_transform`.

**Suggestions:**
- Run UMAP using the function `umap_model.fit_transform`.
- Plot the result using `visualize_components`.

In [ ]:
help(umap_model.fit_transform)

In [ ]:
#################################################
## TODO for students
# Comment once you've completed the code
raise NotImplementedError("Student exercise: perform UMAP")
#################################################

# Perform UMAP
embed = ...

# Visualize the data
visualize_components(..., ..., labels)

In [ ]:
# to_remove solution

# Perform UMAP
embed = umap_model.fit_transform(X)

# Visualize the data
with plt.xkcd():
  visualize_components(embed[:, 0], embed[:, 1], labels)

## Coding Exercise 2.2: Run UMAP with different neighborhood sizes

Unlike PCA, UMAP has parameters that affect the visualization. One important parameter is `n_neighbors`, which controls how local or global the embedding is. Smaller values emphasize very local neighborhoods; larger values preserve broader structure.

**Steps:**
- Rerun UMAP with `n_neighbors` values of 5, 15, and 50.
- Keep `n_components=2`, `min_dist=0.1`, and `random_state=2020`.

In [ ]:
def explore_neighbors(values, X, labels):
  """
  Plots UMAP embeddings using different neighborhood sizes.

  Args:
    values (list of int) : n_neighbors values to visualize
    X (np.ndarray of floats) : matrix with the dataset
    labels (np.ndarray of int) : array with the labels

  Returns:
    Nothing.

  """
  for n_neighbors in values:

    #################################################
    ## TO DO for students: redefine the UMAP model with this n_neighbors value,
    ## then fit it on X and plot the embedding.
    # Comment these lines when you complete the function
    raise NotImplementedError("Student Exercise! Explore UMAP with different n_neighbors values")
    #################################################

    # Perform UMAP
    umap_model = ...

    embed = umap_model.fit_transform(X)
    visualize_components(embed[:, 0], embed[:, 1], labels, show=False)
    plt.title(f"n_neighbors: {n_neighbors}")


# Visualize
values = [5, 15, 50]
explore_neighbors(values, X, labels)